In [ ]:
pip install transformers torchaudio indic_transliteration edge_tts asyncio elevenlabs pykokoro soundfile pyannote.audio faster_whisper sentence_transformers numpy


AUDIO UPLOAD


In [ ]:
from google.colab import files
u=files.upload()

PREPROCESSING AUDIO

In [ ]:
import torchaudio
import torchaudio.transforms as T

waveform,sr=torchaudio.load("newfile.wav")

if waveform.shape[0]>1:
    waveform=waveform.mean(dim=0)
else:
  waveform=waveform.squeeze()

target_rate=16000
if sr!=target_rate:
    resample=T.Resample(sr,target_rate)
    waveform=resample(waveform)
    sr=target_rate

waveform=waveform/waveform.abs().max()
print(waveform.shape)

In [ ]:
from IPython.display import Audio

Audio(waveform,rate=sr)

TRANSCRIPTION

In [ ]:
from transformers import WhisperProcessor,WhisperForConditionalGeneration
import torch

processor=WhisperProcessor.from_pretrained("openai/whisper-small")
model=WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

chunks=[]
step_size=30*sr
overlap_len=5*sr

device="cuda" if torch.cuda.is_available() else "cpu"
model=model.to(device)

for i in range(0,waveform.shape[0],step_size-overlap_len):
    chunk=waveform[i:i+step_size]
    chunks.append(chunk)

text=""
for chunk in chunks:
    inputs=processor(chunk,sampling_rate=16000,return_tensors="pt",return_attention_mask=True)
    input_features=inputs["input_features"].to(device)

    with torch.no_grad():
      predicted_ids=model.generate(input_features)

    transcription=processor.batch_decode(predicted_ids,skip_special_tokens=True)[0]

    text+=transcription


print(text)

DIARIZATION

In [ ]:
!ffmpeg -i newfile.wav -ar 16000 -ac 1 meeting.wav

In [ ]:
from faster_whisper import WhisperModel
from pyannote.audio import Pipeline
import torch
from google.colab import userdata

HF_TOKEN =  userdata.get("HF_TOKEN")

device = "cuda" if torch.cuda.is_available() else "cpu"

# ----------------------------
# Whisper Transcription
# ----------------------------
whisper = WhisperModel(
    "small",
    device=device,
    compute_type="float16" if device == "cuda" else "int8"
)

segments, info = whisper.transcribe(
    "meeting.wav",
    beam_size=5
)

transcript_segments = []

for segment in segments:
    transcript_segments.append({
        "start": segment.start,
        "end": segment.end,
        "text": segment.text.strip()
    })

# ----------------------------
# Speaker Diarization
# ----------------------------
pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)

if device == "cuda":
    pipeline.to(torch.device("cuda"))

diarization = pipeline("meeting.wav")

annotation= diarization.speaker_diarization
speaker_segments = []

for turn, _, speaker in annotation.itertracks(yield_label=True):
    speaker_segments.append({
        "start": turn.start,
        "end": turn.end,
        "speaker": speaker
    })

# ----------------------------
# Match transcript to speaker
# ----------------------------
final_transcript = []

for t in transcript_segments:

    best_speaker = "UNKNOWN"
    max_overlap = 0

    for s in speaker_segments:

        overlap = max(
            0,
            min(t["end"], s["end"]) -
            max(t["start"], s["start"])
        )

        if overlap > max_overlap:
            max_overlap = overlap
            best_speaker = s["speaker"]

    final_transcript.append(
        f"{best_speaker}: {t['text']}"
    )

# ----------------------------
# Output
# ----------------------------
result = "\n".join(final_transcript)

print(result)

LLM INTEGRATION

In [ ]:
from google import genai
from dotenv import load_dotenv
import time
from google.genai.errors import APIError
from google.colab import userdata


api_key=userdata.get("GEMINI_API_KEY")
client=genai.Client(api_key=api_key)

prompt=f'''
You are an AI Meeting Intelligence Assistant.

Analyze the following meeting transcript and provide:

1. Executive Summary
   - 5-10 concise bullet points covering the main discussion.

2. Action Items
   - Task
   - Owner (if mentioned)
   - Deadline (if mentioned)

3. Key Decisions
   - List all decisions that were finalized during the meeting.

4. Risks and Concerns
   - Potential risks, blockers, concerns, or unresolved issues discussed.

5. Open Questions
   - Questions that were raised but not answered or finalized.

Instructions:
- Use only information present in the transcript.
- Do not invent details.
- If a section has no relevant information, write "Not mentioned."
- Keep the output clear and structured.

{result}
'''


# 2. Wrapped in a retry loop to instantly bypass the 503 and 429 errors
for attempt in range(3):
    try:
        response = client.models.generate_content(
            # 3. Upgraded to 2.5-flash for the fastest response and lowest failure rate
            model="gemini-2.5-flash",
            contents=prompt
        )
        print(response.text)
        break # Success! Break out of the retry loop.

    except APIError as e:
        if e.code in [429, 503] and attempt < 2:
            print(f"Server busy or rate limited ({e.code}). Retrying in 5 seconds...")
            time.sleep(5)
        else:
            print(f"An unexpected error occurred: {e}")
            raise e

RETRIEVAL AUGMENTED GENERATION(RAG)


In [ ]:
!pip install faiss-cpu

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time
from google.genai.errors import APIError
#CHUNKING

chunks=[]
chunk_size=5
current=[]

for line in result.split("\n"):
  current.append(line)

  if len(current)==chunk_size:
    chunks.append(" ".join(current))
    current=[]


if current:
   chunks.append(" ".join(current))


#embeddings
model=SentenceTransformer("all-MiniLM-L6-v2")
embeddings=model.encode(chunks)
embeddings=np.array(embeddings).astype("float32")

#store in vector db
index=faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)


#query handling
query="What is the main positioning of the product"
query_emb=model.encode([query]).astype("float32")

D,I=index.search(np.array(query_emb),k=3)
retrieved_chunks=[chunks[i] for i in I[0]]

context="\n".join(retrieved_chunks)

prompt=f'''

You are an AI meeting assisstant

context:
{context}


Question:
{query}

Answer clearly and concicesly.
'''

for attempt in range(3):
    try:
        response = client.models.generate_content(
            # 3. Upgraded to 2.5-flash for the fastest response and lowest failure rate
            model="gemini-2.5-flash",
            contents=prompt
        )
        print(response.text)
        break # Success! Break out of the retry loop.

    except APIError as e:
        if e.code in [429, 503] and attempt < 2:
            print(f"Server busy or rate limited ({e.code}). Retrying in 5 seconds...")
            time.sleep(5)
        else:
            print(f"An unexpected error occurred: {e}")
            raise e


LANGCHAIN

In [ ]:
!pip install langchain langchain-text-splitters langchain_community faiss-cpu

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
import time
from google.genai.errors import APIError

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

chunks=text_splitter.split_text(result)

embeddings_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

vectorstore=FAISS.from_texts(chunks,embeddings_model)
query="Who is responsible for handling mobile ui design"
docs=vectorstore.similarity_search(query,k=3)

context="\n".join([doc.page_content for doc in docs])

prompt=f'''

You are an AI meeting assisstant

context:
{context}


Question:
{query}

Answer clearly and concicesly.
'''

for attempt in range(3):
    try:
        response = client.models.generate_content(
            # 3. Upgraded to 2.5-flash for the fastest response and lowest failure rate
            model="gemini-2.5-flash",
            contents=prompt
        )
        print(response.text)
        break # Success! Break out of the retry loop.

    except APIError as e:
        if e.code in [429, 503] and attempt < 2:
            print(f"Server busy or rate limited ({e.code}). Retrying in 5 seconds...")
            time.sleep(5)
        else:
            print(f"An unexpected error occurred: {e}")
            raise e